# Phase 4 — Phylogenetic Comparison (Kmult primary test)

Runs `src/phylo_comparison/` (Python: builds Kmult-admissible feature matrices per pattern dimension + exports the pruned tree with canonical species-key tip labels) followed by `r/phase4_kmult.R` (R: `geomorph::physignal.z` per dimension, `compare.physignal.z` across dimensions, multiple-comparisons correction) via rpy2's `%%R` cell magic, so both halves run in one notebook without switching runtimes.

**No GPU needed — use a CPU runtime.**

**Prerequisite:** Phase 3 must already have produced `reports/species_features.csv` — run `Phase3_Distance_Matrices.ipynb` first.

**IMPORTANT — read before trusting any result this notebook produces:** `r/phase4_kmult.R` has never been executed or verified against a real R installation — unlike every other piece of code in this project, it has not yet been checked against real output. Its own header comment says the same thing. Section 6 below runs a smoke test against `geomorph`'s own bundled `plethspecies` example **before** touching real data — scroll up and actually look at that output (compare it against the `geomorph` manual's own documented example) before trusting anything Section 7 produces. See README.md's Planned Approach (step 4) and the v4.1.0 changelog entry for the full design reasoning.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Open the same Drive-resident project Phase 1-3 used

Must resolve to the same `PROJECT_DIR` Phase 3 wrote `reports/species_features.csv` into.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/guptrishi01/Surgeonfish_Neural_Network_Phylogenetics.git"
PROJECT_DIR = Path("/content/drive/MyDrive/Surgeonfish_Neural_Network_Phylogenetics")

if not PROJECT_DIR.exists():
    print(f"Cloning into {PROJECT_DIR} (first time - pulls ~1.8GB, be patient)...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"{PROJECT_DIR} already exists - pulling latest code only.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

In [ ]:
%cd {PROJECT_DIR}

## 3. Install Python dependencies and add the local package to the path

`biopython` is reused from Phase 3 (tree pruning); `phylo_comparison` has no further Python dependencies beyond it, numpy, and scipy.

In [ ]:
%pip install -q "biopython>=1.81"

import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import Bio, numpy, scipy
print("Deps OK - biopython", Bio.__version__, "numpy", numpy.__version__, "scipy", scipy.__version__)

## 4. Confirm Phase 3's output is actually there

In [ ]:
features_path = PROJECT_DIR / "reports" / "species_features.csv"
if not features_path.exists():
    print(f"Nothing at {features_path} yet - run Phase3_Distance_Matrices.ipynb first.")
else:
    n_lines = sum(1 for _ in open(features_path, encoding="utf-8"))
    print(f"Found {features_path} ({n_lines - 1} species row(s)).")

## 5. Run the Phase 4 Python preparation

Builds each dimension's Kmult-admissible feature matrix (Smithson-Verkuilen + logit for the three true proportions-of-count, `log1p` for everything else, then standardized) and exports the pruned tree with tip labels renamed to canonical species keys — see `src/phylo_comparison/__init__.py` and the v4.1.0 changelog entry for the full reasoning. Fails loudly (`ValueError`) if any dimension's matrix is ill-conditioned rather than silently proceeding.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s", force=True
)

from phylo_comparison.config import ExportConfig, FeaturePrepConfig
from phylo_comparison.pipeline import run as run_phase4_prep

feature_prep_config = FeaturePrepConfig(species_features_csv_path=features_path)
export_config = ExportConfig(
    output_dir=PROJECT_DIR / "outputs" / "phase4",
    tree_path=PROJECT_DIR / "data" / "phylogeny" / "actinopt_12k_treePL.tre",
    species_coverage_csv_path=PROJECT_DIR / "data" / "phylogeny" / "species_coverage.csv",
)

species_order = run_phase4_prep(feature_prep_config, export_config)
print(f"\n{len(species_order)} species in the Phase 4 analysis set.")

## 6. Inspect the exported files before handing off to R

In [ ]:
phase4_dir = PROJECT_DIR / "outputs" / "phase4"
for f in sorted(phase4_dir.iterdir()):
    print(f.name, f"({f.stat().st_size} bytes)")

import csv
with open(phase4_dir / "color_kmult_features.csv", newline="", encoding="utf-8") as fh:
    rows = list(csv.reader(fh))
print("\ncolor_kmult_features.csv header + first 3 rows:")
for row in rows[:4]:
    print(row)

## 6b. Prepare the sensitivity run (47 species)

README.md's own Phase 4 verification criteria require the sparse-species sensitivity check to be reviewed before the feature matrices feeding these tests are trusted. This regenerates Phase 3's `min_images_per_species=5` run (dropping *Acanthurus triostegus* and *Naso tuberosus*, 49 → 47 species) and prepares it for Kmult, into separate directories so the primary run's files are never overwritten. Regenerated here rather than assumed present, so this notebook doesn't depend on whether Phase 3's optional step 7 happened to be run in this particular Drive folder.

A result that only holds at exactly one species-count cutoff would be a much weaker finding than one that survives dropping the two noisiest species — the R script compares the two runs' significance verdicts directly at the end.

In [ ]:
from distance_matrices.config import (
    AggregationConfig, DistanceMatrixConfig, PhylogenyConfig,
)
from distance_matrices.pipeline import run as run_phase3

sens_features_csv = PROJECT_DIR / "reports" / "species_features_min5.csv"
sens_distance_dir = PROJECT_DIR / "outputs" / "sensitivity_min5"

phase3_sens = run_phase3(
    AggregationConfig(output_csv_path=sens_features_csv, min_images_per_species=5),
    DistanceMatrixConfig(output_dir=sens_distance_dir),
    PhylogenyConfig(output_path=sens_distance_dir / "patristic_distance_matrix.csv"),
)
print(f"Phase 3 sensitivity run: {len(phase3_sens.species_order)} species")

sens_export_config = ExportConfig(
    output_dir=PROJECT_DIR / "outputs" / "phase4_min5",
    tree_path=PROJECT_DIR / "data" / "phylogeny" / "actinopt_12k_treePL.tre",
    species_coverage_csv_path=PROJECT_DIR / "data" / "phylogeny" / "species_coverage.csv",
)
sens_species = run_phase4_prep(
    FeaturePrepConfig(species_features_csv_path=sens_features_csv), sens_export_config
)
print(f"Phase 4 sensitivity prep: {len(sens_species)} species")

## 7. Install R dependencies and load the `%%R` cell magic

`rpy2` ships preinstalled on Colab; this just loads its IPython extension. `geomorph` does not ship preinstalled and compiles from source on first install — this can take **5-15 minutes**, be patient. Re-running this cell on a later session of the same runtime is a no-op if `geomorph` is already installed.

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
if (!requireNamespace("geomorph", quietly = TRUE)) {
  install.packages("geomorph", dependencies = TRUE, Ncpus = 2)
}
library(geomorph)
packageVersion("geomorph")

## 8. Run `r/phase4_kmult.R`

This sources the actual, version-controlled script — not a copy pasted into the notebook — so there is exactly one place this logic lives. The script runs the whole analysis twice through the *same* function: once for the primary 49-species set, once for the 47-species sensitivity set prepared above, then prints a side-by-side comparison of whether each dimension's significance verdict is stable.

Steps 0-8 have all been run and verified on real data for the primary set (see the v4.1.1 / v4.2.1 changelog entries — two real bugs were caught that way, a guessed geomorph field name and a degenerate Mantel null). The sensitivity re-run and the comparison table at the end are new. **Read the comparison output rather than just collecting it** — a verdict that flips when two species are dropped is a real finding about the result's fragility, not a glitch to route around.

In [ ]:
project_dir_str = str(PROJECT_DIR)

In [ ]:
%%R -i project_dir_str
setwd(project_dir_str)
source("r/phase4_kmult.R")

## 9. Read the results back into Python

A null result (non-significant, BH-corrected p) is **inconclusive, not evidence of no association** — Harmon & Glor (2010) found real power limitations even with the correct phylogenetic-signal remedy applied. See README.md.

Both the primary Kmult results and the secondary Mantel results should be reviewed together before calling anything about Phase 4 final.

In [ ]:
import pandas as pd

kmult_results = pd.read_csv(phase4_dir / "kmult_results.csv")
mantel_results = pd.read_csv(phase4_dir / "mantel_results.csv")
sensitivity = pd.read_csv(phase4_dir / "sensitivity_comparison.csv")

print("Kmult (primary, 49 species):")
display(kmult_results)
print("\nMantel (secondary, standard label-permutation):")
display(mantel_results)
print("\nSensitivity: 49 species vs 47 (sparse species dropped):")
display(sensitivity)